# Testing Cupy Functionality?

In [4]:
import numpy as np
import cupy as cp
import cupyx as cpx
import numba
from numba import cuda
from numba.experimental import jitclass

In [9]:
spec = [
	('x', numba.float64),
	('y', numba.float64),
	('z', numba.float64),
]

@jitclass(spec)
class double3(object):
	def __init__(self):
		self.x = 0.0
		self.y = 0.0
		self.z = 0.0

In [60]:
@cuda.jit
def testingClasses(a, b, c):
	thread_id = cuda.grid(1)
	for i in range(a.shape[0]):
		c[i] = a[i] + b[i]

In [61]:
a = cp.ones(3)
b = cp.ones(3)
c = cp.zeros(3)
testingClasses[1, 1](a, b, c)
print(a, b, c)

[1. 1. 1.] [1. 1. 1.] [2. 2. 2.]


/home/david/anaconda3/lib/python3.9/site-packages/numba/cuda/compiler.py:726: NumbaPerformanceWarning: Grid size (1) < 2 * SM count (32) will likely result in GPU under utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


In [36]:
test = double3()

In [10]:
print(test)

In [7]:
@cuda.jit
def genRandomUniform(rng_states, iterations, out):
	"""Find the maximum value in values and store in result[0]"""
	thread_id = cuda.grid(1)
	# Compute pi by drawing random (x, y) points and finding what
	# fraction lie inside a unit circle
	x = xoroshiro128p_uniform_float32(rng_states, thread_id)
	out[thread_id] = x

In [33]:
threads_per_block = 64
blocks = 1024
rng_states = create_xoroshiro128p_states(threads_per_block * blocks, seed=1)
out = cp.zeros(threads_per_block * blocks, dtype=np.float32)
start = time.time()
for i in range(1000):
	genRandomUniform[blocks, 64](rng_states, 1, out)
stop = time.time()
print(stop-start)

0.5359477996826172


In [32]:
out2 = cp.zeros(threads_per_block * blocks, dtype=np.float32)
start = time.time()
for i in range(1000):
	out2 = cp.random.uniform(size=threads_per_block*blocks, dtype=cp.float32)[:]
stop = time.time()
print(stop-start)

0.028252601623535156


In [28]:
out2

array([0., 0., 0., ..., 0., 0., 0.], dtype=float32)

In [181]:
@cpx.jit.rawkernel(device=True, mode='cuda')
def gradient2(pos):
	return int(pos)

@cpx.jit.rawkernel()
def test2(x, size):
	tid = jit.blockIdx.x * jit.blockDim.x + jit.threadIdx.x
	gen = cp.random.Philox4x3210(seed=int(tid))
	if tid < size:
		x[tid] = float(cp.random.uniform())

x = cp.zeros(10, dtype=np.float64)
test2[1, 10](x, 10)  # RawKernel style

/home/david/anaconda3/lib/python3.9/site-packages/cupyx/jit/_interface.py:171: FutureWarning: cupyx.jit.rawkernel is experimental. The interface can change in the future.
  cupy._util.experimental('cupyx.jit.rawkernel')


TypeError: Invalid function call 'Philox4x3210'.

  @cpx.jit.rawkernel()
  def test2(x, size):
  	tid = jit.blockIdx.x * jit.blockDim.x + jit.threadIdx.x
> 	gen = cp.random.Philox4x3210(seed=int(tid))
  	if tid < size:
  		x[tid] = float(cp.random.uniform())


In [200]:
gen = cp.cuda.curand.createGenerator(cp.cuda.curand.CURAND_RNG_PSEUDO_XORWOW)
cp.cuda.curand.generate?

Docstring: generate(size_t generator, size_t outputPtr, size_t num)
Type:      builtin_function_or_method


In [145]:
test = cp.random.XORWOW(seed=cp.arange(10))
gen = cp.random.Generator(test)


@cp.fuse()
def test3(x, size):
	tid = jit.blockIdx.x * jit.blockDim.x + jit.threadIdx.x
	a = cp.random.XORWOW()
	gen = cp.random.Generator(a)
	if tid < size:
		x[tid] = float(gen.uniform())

x = cp.zeros(10, dtype=np.float64)
test3(x, 10)  # RawKernel style

AttributeError: 'Data' object has no attribute 'x'

In [63]:
cp.random.XORWOW(seed=0, size=10)

In [64]:
cpx.curand?

Object `cpx.curand` not found.


In [77]:
cp.cuda.curand.createGenerator(101)

127508224

In [79]:
cp.cuda.curand.

Docstring: generate(size_t generator, size_t outputPtr, size_t num)
Type:      builtin_function_or_method


In [81]:
cp.random.Generator(0)